# Create Genie Space — Enterprise Sales Analytics

Adds table and column descriptions to Unity Catalog, then creates (or updates) a Genie Space so sales leaders can explore the data with natural language.

**Prerequisite:** Run `generate_sales_data` first to populate the Delta tables in `{CATALOG}.{SCHEMA}`.

The next two cells upgrade the Databricks SDK to the latest version (the pre-installed version on serverless compute doesn't include the Genie API) and restart Python so the new version is loaded.

In [ ]:
%pip install --upgrade databricks-sdk --quiet

In [ ]:
dbutils.library.restartPython()

## 1. Configuration

In [ ]:
CATALOG = "tabpfn_databricks"
SCHEMA = "agent"

## 2. Add Table & Column Descriptions

Enriches Unity Catalog metadata using `COMMENT ON TABLE` and `ALTER TABLE ... ALTER COLUMN ... COMMENT` statements. Genie relies on these descriptions to understand the data and generate accurate SQL, so well-annotated tables are essential for good query generation.

In [ ]:
TABLE_DESCRIPTIONS = {
    "accounts": {
        "table": "Firmographic data for prospect and customer accounts. Each row is one company, segmented as High-Growth Tech, Fortune 500, or Mid-Market. Each account is assigned to exactly one sales rep.",
        "columns": {
            "account_id": "Primary key (e.g. ACCT-0001)",
            "account_name": "Company legal name",
            "segment": "Market segment: High-Growth Tech, Fortune 500, or Mid-Market",
            "industry": "Vertical such as SaaS, Financial Services, Education, etc.",
            "employee_count": "Approximate headcount",
            "region": "Geographic region: North America, Europe, APAC, or LATAM",
            "annual_revenue_mm": "Estimated annual revenue in millions USD",
            "annual_acv_target": "Expected annual platform spend in USD, derived from employee count and segment-specific per-employee rate",
            "created_date": "Date the account was first entered into the CRM",
        },
    },
    "sales_reps": {
        "table": "Sales team roster with quota targets and attainment. Quota attainment follows a Beta distribution producing realistic variance (some reps at 60%, others at 120%+).",
        "columns": {
            "rep_id": "Primary key (e.g. REP-001)",
            "rep_name": "Full name of the sales representative",
            "team": "Sales team assignment (e.g. Enterprise West, Growth)",
            "hire_date": "Date the rep joined the company",
            "annual_quota": "Annual booking target in USD",
            "quota_attainment_pct": "Percentage of annual quota achieved (40-140 range)",
        },
    },
    "products": {
        "table": "Product catalog of 8 data and AI platform offerings. Includes the base Unified Lakehouse Platform plus add-ons (SQL Compute, Streaming, Governance, BI) and premium tiers (ML Studio Pro, GenAI Agent Builder). Used for cross-sell and upsell analysis.",
        "columns": {
            "product_id": "Primary key (e.g. PROD-001)",
            "product_name": "Commercial product name (e.g. Unified Lakehouse Platform, ML Studio Pro, GenAI Agent Builder)",
            "tier": "Product tier: Base, Add-on, Premium, or Service",
            "list_acv": "List annual contract value in USD before discounts",
            "category": "Functional category: Platform, Compute, AI/ML, Data Engineering, Governance, Analytics, Support",
        },
    },
    "opportunities": {
        "table": "Sales pipeline with one row per deal. Includes ACV, stage progression, lead source, close-date projections, and a promotion flag. Volume exhibits Q4 seasonality trough and Q1 budget-flush spike. ~10% of opportunities have a promotion applied.",
        "columns": {
            "opportunity_id": "Primary key (e.g. OPP-00001)",
            "account_id": "Foreign key to accounts.account_id",
            "rep_id": "Foreign key to sales_reps.rep_id",
            "lead_source": "How the deal originated: Outbound, Inbound, or Partner",
            "acv": "Annual contract value in USD (log-normal, segment-dependent). May be reduced by Discount promotions.",
            "stage": "Current pipeline stage: Discovery, Demo, Negotiation, Closed/Won, or Closed/Lost",
            "created_date": "Date the opportunity was created",
            "close_date": "Actual or projected close date",
            "days_in_pipeline": "Number of days from creation to close. May be shortened by Delivery Support promotions.",
            "has_promotion": "Boolean flag: true if a promotion was applied to this opportunity",
        },
    },
    "opportunity_products": {
        "table": "Line items linking opportunities to products. Every deal includes Unified Lakehouse Platform; add-ons and premium products are attached probabilistically based on account segment. Enables cross-sell/upsell analysis.",
        "columns": {
            "opp_product_id": "Primary key (e.g. OPRD-000001)",
            "opportunity_id": "Foreign key to opportunities.opportunity_id",
            "product_id": "Foreign key to products.product_id",
            "line_acv": "Discounted ACV for this line item in USD",
            "discount_pct": "Discount percentage applied (0-20%)",
        },
    },
    "promotions": {
        "table": "Promotions applied to opportunities. At most one promotion per opportunity (~10% of deals). Three types: Discount (reduces ACV, boosts win rate), Delivery Support (shortens pipeline, slight win-rate boost), and Enablement (strongest win-rate boost). ~65% of promotions have a measurable effect on deal outcomes.",
        "columns": {
            "promotion_id": "Primary key (e.g. PROMO-00001)",
            "opportunity_id": "Foreign key to opportunities.opportunity_id. One-to-one relationship.",
            "promotion_type": "Type of promotion: Discount, Delivery Support, or Enablement",
            "applied_date": "Date the promotion was applied to the deal",
            "had_effect": "Boolean flag: true if the promotion had a measurable effect on ACV, pipeline velocity, or win rate",
        },
    },
    "sales_activities": {
        "table": "Logged sales touches per opportunity: emails, meetings, proof-of-concept engagements, and enablement sessions. Activity volume correlates with deal stage depth and rep attainment. Enablement sessions are added for opportunities with active Enablement promotions. Use for activity-vs-win-rate analysis.",
        "columns": {
            "activity_id": "Primary key (e.g. ACT-0000001)",
            "opportunity_id": "Foreign key to opportunities.opportunity_id",
            "rep_id": "Foreign key to sales_reps.rep_id",
            "activity_type": "Type of touch: Email, Meeting, POC, or Enablement",
            "activity_date": "Date the activity occurred",
            "duration_minutes": "Duration in minutes (set for meetings and enablement sessions; null for emails and POCs)",
            "poc_days": "Length of proof-of-concept in days (null for emails, meetings, and enablement sessions)",
        },
    },
}

for table_name, meta in TABLE_DESCRIPTIONS.items():
    fqn = f"{CATALOG}.{SCHEMA}.{table_name}"
    spark.sql(f"COMMENT ON TABLE {fqn} IS '{meta['table']}'")
    for col, desc in meta["columns"].items():
        spark.sql(f"ALTER TABLE {fqn} ALTER COLUMN {col} COMMENT '{desc}'")
    print(f"  {fqn}: table + {len(meta['columns'])} column descriptions applied")

print("\nDone — all descriptions applied.")

## 3. Create Genie Space

Creates (or updates) a Genie Space pointing at all six tables so sales leaders can explore the data with natural language.

The Genie REST API expects a `serialized_space` JSON string (version 2) containing:
- **Sample questions** — each with a 32-char hex ID, sorted alphabetically by ID. These appear in the Genie UI to guide users.
- **Table identifiers** — three-level `catalog.schema.table` names, sorted alphabetically.

The cell also finds a **serverless SQL warehouse** (Genie needs one to execute its generated queries) and is **idempotent**: if a space with the same title already exists it updates it, otherwise it creates a new one.

In [ ]:
import json, hashlib
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

GENIE_SPACE_NAME = "Enterprise Sales Analytics"
GENIE_DESCRIPTION = (
    f"Explore enterprise software sales data across a 24-month period.\n\n"
    f"Tables ({CATALOG}.{SCHEMA}):\n"
    f"  - accounts: Firmographic data for 500 accounts across 3 segments, each with an annual ACV target\n"
    f"  - sales_reps: 30 reps with realistic quota-attainment variance\n"
    f"  - products: 8 data & AI platform products (Lakehouse, SQL Compute, ML Studio, Streaming, Governance, GenAI, BI, Support)\n"
    f"  - opportunities: ~3 500 deals with ACV, stage, lead source, promotion flag. Each account is assigned to exactly one rep.\n"
    f"  - promotions: ~350 promotions (Discount, Delivery Support, Enablement) applied to opportunities; ~65% have a measurable effect\n"
    f"  - opportunity_products: Line items for cross-sell/upsell analysis. Every deal includes Unified Lakehouse Platform.\n"
    f"  - sales_activities: ~65 000 emails, meetings, POC, and enablement session records\n\n"
    f"Key relationships:\n"
    f"  opportunities.account_id -> accounts.account_id\n"
    f"  opportunities.rep_id -> sales_reps.rep_id\n"
    f"  promotions.opportunity_id -> opportunities.opportunity_id (one-to-one)\n"
    f"  opportunity_products.opportunity_id -> opportunities.opportunity_id\n"
    f"  opportunity_products.product_id -> products.product_id\n"
    f"  sales_activities.opportunity_id -> opportunities.opportunity_id\n"
    f"  sales_activities.rep_id -> sales_reps.rep_id"
)

SAMPLE_QUESTIONS = [
    "What is the overall win rate by lead source?",
    "Show me the top 10 reps by total closed-won ACV",
    "What is the average deal size by account segment?",
    "How does opportunity volume trend month over month?",
    "Which products are most commonly cross-sold with Unified Lakehouse Platform?",
    "What is the average number of meetings for won vs lost deals?",
    "Show accounts with the most open pipeline value",
    "Which accounts have closed-won ACV exceeding their annual ACV target?",
    "What is the win rate for deals with promotions vs without?",
    "How does each promotion type (Discount, Delivery Support, Enablement) affect win rate?",
]

TABLE_NAMES = [
    "accounts", "opportunity_products", "opportunities",
    "products", "promotions", "sales_activities", "sales_reps",
]

def _hex_id(seed: str) -> str:
    return hashlib.md5(seed.encode()).hexdigest()

sample_q_objects = sorted(
    [{"id": _hex_id(f"sq-{i}"), "question": [q]} for i, q in enumerate(SAMPLE_QUESTIONS)],
    key=lambda x: x["id"],
)

table_objects = sorted(
    [{"identifier": f"{CATALOG}.{SCHEMA}.{t}"} for t in TABLE_NAMES],
    key=lambda x: x["identifier"],
)

serialized = json.dumps({
    "version": 2,
    "config": {"sample_questions": sample_q_objects},
    "data_sources": {"tables": table_objects},
})

wh = next(
    (wh for wh in w.warehouses.list() if wh.enable_serverless_compute),
    None,
)
if not wh:
    raise RuntimeError("No serverless SQL warehouse found")
warehouse_id = wh.id
print(f"Using warehouse: {wh.name} ({warehouse_id})")

existing = w.genie.list_spaces()
space_id = None
for s in (existing.spaces or []):
    if s.title == GENIE_SPACE_NAME:
        space_id = s.space_id
        break

if space_id:
    w.genie.update_space(
        space_id=space_id,
        title=GENIE_SPACE_NAME,
        description=GENIE_DESCRIPTION,
        serialized_space=serialized,
        warehouse_id=warehouse_id,
    )
    print(f"Updated Genie Space: {GENIE_SPACE_NAME} (space_id={space_id})")
else:
    result = w.genie.create_space(
        warehouse_id=warehouse_id,
        serialized_space=serialized,
        title=GENIE_SPACE_NAME,
        description=GENIE_DESCRIPTION,
    )
    space_id = result.space_id
    print(f"Created Genie Space: {GENIE_SPACE_NAME} (space_id={space_id})")

host = spark.conf.get("spark.databricks.workspaceUrl")
print(f"\n  https://{host}/genie/rooms/{space_id}")

## 4. Ask Questions

The `ask()` helper below uses the Genie Conversation API:
- **New conversation** — `start_conversation_and_wait()` sends a question and blocks until Genie generates and executes SQL.
- **Follow-up** — passing a `conversation_id` to `create_message_and_wait()` lets Genie use context from earlier messages (e.g. "break that down by region").
- **Results** — fetched via `get_message_attachment_query_result()`, which returns a `StatementResponse` (same format as the Statement Execution API), converted to a pandas DataFrame for display.

In [ ]:
import pandas as pd
from databricks.sdk.service.dashboards import MessageStatus

def ask(question, conversation_id=None):
    """Ask a question to the Genie Space and display the generated SQL + results."""
    print(f"\n{'─'*80}")
    print(f"Q: {question}")
    print(f"{'─'*80}")

    if conversation_id is None:
        resp = w.genie.start_conversation_and_wait(space_id=space_id, content=question)
        conversation_id = resp.conversation_id
    else:
        resp = w.genie.create_message_and_wait(
            space_id=space_id, conversation_id=conversation_id, content=question,
        )

    message_id = resp.message_id

    if resp.status == MessageStatus.COMPLETED:
        for att in (resp.attachments or []):
            if att.query:
                print(f"\nSQL:\n{att.query.query}")
            if att.query and att.attachment_id:
                try:
                    qr = w.genie.get_message_attachment_query_result(
                        space_id=space_id,
                        conversation_id=conversation_id,
                        message_id=message_id,
                        attachment_id=att.attachment_id,
                    )
                    sr = qr.statement_response
                    if sr and sr.manifest and sr.result and sr.result.data_array:
                        cols = [c.name for c in sr.manifest.schema.columns]
                        df = pd.DataFrame(sr.result.data_array, columns=cols)
                        print(f"\nResult ({len(df)} rows):")
                        display(df)
                except Exception as e:
                    print(f"\n(Could not fetch query result: {e})")
            if att.text:
                print(f"\nGenie says: {att.text.content}")
    else:
        status = resp.status.value if resp.status else "UNKNOWN"
        print(f"Status: {status}")
        for att in (resp.attachments or []):
            if att.text:
                print(f"Genie says: {att.text.content}")

    return conversation_id

In [ ]:
conv_id = ask("What is the overall win rate by lead source?")

In [ ]:
ask("Show me the top 5 reps by total closed-won ACV")

In [ ]:
ask("How does opportunity volume trend month over month — is there a Q4 dip?")

In [ ]:
conv_id = ask("What is the average number of meetings for won vs lost deals?")
ask("Break that down by account segment", conversation_id=conv_id)